# **LightGBM**

In [1]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [2]:
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep, append_results, eval_thresholds
from preprocessing.target import ttp_target, hybrid_target
from metrics.Metrics import merged_metrics
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

In [3]:
from lightgbm import LGBMClassifier


def train_lgbm_ttp(
    df,
    train_size,
    test_size,
    step,
):

    splitter = prep(
        df=df,
        target_fn=ttp_target,
        target_name="ttp",
        target_col="TTP_class",
        horizons=[12, 24, 48],
        train_size=train_size,
        test_size=test_size,
        step=step,
        target_kwargs={"n_classes": 3},
        scale_cols=[
            "Open", "High", "Low", "Close",
            "Alligator_Jaw", "Alligator_Teeth", "Alligator_Lips",
            "AO",
            "AddOn_Anchor_Level", "AddOn_Size_Pct"
        ]
    )

    for X_train, X_test, y_train, y_test, scaler in splitter:

        model = LGBMClassifier(
            objective="multiclass",
            num_class=3,
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        metrics = merged_metrics(y_test, y_pred)

        append_results({
            "task_type": "classification",

            "model_name": "LightGBM",
            "model_family": "lgbm",
            "model_params": {
                "n_estimators": 300,
                "max_depth": 5,
                "learning_rate": 0.05,
                "subsample": 0.8,
                "colsample_bytree": 0.8
            },

            "target_name": "ttp",
            "target_variant": "3class",
            "horizons": "12_24_48",

            **metrics
        })

train_lgbm_ttp(
    df=df,
    train_size=1000,
    test_size=200,
    step=100
)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000668 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2848
[LightGBM] [Info] Number of data points in the train set: 1000, number of used features: 31
[LightGBM] [Info] Start training from score -0.473209
[LightGBM] [Info] Start training from score -2.577022
[LightGBM] [Info] Start training from score -1.200645
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further